# Bloque 3 - Análisis Exploratorio + Experimentación
**Prueba Técnica - Data Analyst - Datos de tetail multiformato Centroamérica**

Este notebook asume que los datos se cargan desde CSVs exportados de BigQuery
(dataset `test-eda-507323.producto`), o desde las vistas creadas en el Bloque 1.

Todas las celdas están listas para correr


In [ ]:
# Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


In [ ]:
# --- OPCIÓN B (alternativa): carga directa desde BigQuery ---
# from google.cloud import bigquery
# client = bigquery.Client(project="test-eda-507323")
# transactions = client.query("SELECT * FROM `test-eda-507323.producto.transactions`").to_dataframe()
# transaction_items = client.query("SELECT * FROM `test-eda-507323.producto.transaction_items`").to_dataframe()
# stores = client.query("SELECT * FROM `test-eda-507323.producto.stores`").to_dataframe()
# products = client.query("SELECT * FROM `test-eda-507323.producto.producto`").to_dataframe()
# vendors = client.query("SELECT * FROM `test-eda-507323.producto.vendors`").to_dataframe()
# store_promotions = client.query("SELECT * FROM `test-eda-507323.producto.store_promotions`").to_dataframe()


In [ ]:
# Filtro base: solo transacciones completadas, según lo decidido en Bloque 0
tx_completed = transactions[transactions["status"] == "COMPLETED"].copy()

# Dataset enriquecido a nivel línea de ítem, reutilizado en varias preguntas
items_full = (
    transaction_items
    .merge(tx_completed[["transaction_id", "transaction_date", "store_id", "loyalty_card", "customer_id"]],
           on="transaction_id", how="inner")
    .merge(products, on="item_id", how="left")
    .merge(stores[["store_id", "format", "country", "region", "size_sqm"]], on="store_id", how="left")
)
items_full["line_amount"] = items_full["unit_price"] * items_full["quantity"]
items_full.head()


---
## Parte A · Pregunta 1 — Estacionalidad por formato

**Pregunta:** ¿Cómo evoluciona el GMV semanal por formato de tienda? ¿Qué formato es
más sensible a la estacionalidad? Identifica los 3 picos y las 3 caídas más
significativas y propón una hipótesis para cada uno.


In [ ]:
weekly_gmv = (
    tx_completed
    .assign(week=lambda d: d["transaction_date"].dt.to_period("W").dt.start_time)
    .merge(stores[["store_id", "format"]], on="store_id", how="left")
    .groupby(["week", "format"], as_index=False)["total_amount"].sum()
    .rename(columns={"total_amount": "gmv_semanal"})
)

fig, ax = plt.subplots()
for fmt, grp in weekly_gmv.groupby("format"):
    ax.plot(grp["week"], grp["gmv_semanal"], label=fmt, marker="o", markersize=3)
ax.set_title("GMV semanal por formato de tienda")
ax.set_xlabel("Semana")
ax.set_ylabel("GMV")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("bloque3_visualizaciones/p1_estacionalidad_por_formato.png", dpi=150)
plt.show()


In [ ]:
# Sensibilidad a estacionalidad: coeficiente de variación (std/mean) por formato
# Un CV más alto = más sensible a la estacionalidad
cv_por_formato = (
    weekly_gmv.groupby("format")["gmv_semanal"]
    .agg(media="mean", desviacion="std")
    .assign(coef_variacion=lambda d: d["desviacion"] / d["media"])
    .sort_values("coef_variacion", ascending=False)
)
cv_por_formato


In [ ]:
# Top 3 picos y 3 caídas (a nivel chain-wide, sumando todos los formatos)
gmv_total_semanal = weekly_gmv.groupby("week", as_index=False)["gmv_semanal"].sum()
gmv_total_semanal["pct_change"] = gmv_total_semanal["gmv_semanal"].pct_change() * 100

top_3_picos = gmv_total_semanal.nlargest(3, "pct_change")
top_3_caidas = gmv_total_semanal.nsmallest(3, "pct_change")

print("TOP 3 PICOS (mayor % de crecimiento semana vs semana anterior):")
display(top_3_picos)
print("\nTOP 3 CAÍDAS:")
display(top_3_caidas)

# TODO: cruzar las fechas de picos/caídas con calendario de feriados centroamericanos
# (Navidad, Semana Santa, Día de la Madre, Black Friday/Buen Fin) para la hipótesis.


**Hallazgos a completar con datos reales:**
- Formato más sensible a estacionalidad: `[completar según coef_variacion más alto]`
- Picos: `[fecha]` → hipótesis: `[ej. Navidad / quincena / campaña de marketing]`
- Caídas: `[fecha]` → hipótesis: `[ej. post-feriado, inicio de mes con menor liquidez]`


---
## Parte A · Pregunta 2 — Pareto de categorías por formato

**Pregunta:** ¿Qué categorías concentran el 80% del GMV? ¿Las categorías líderes en
HIPERMERCADO son las mismas que en DESCUENTO? ¿Qué dice esto del perfil del comprador?


In [ ]:
gmv_por_categoria_formato = (
    items_full.groupby(["format", "category"], as_index=False)["line_amount"].sum()
    .rename(columns={"line_amount": "gmv"})
)

def pareto(df_formato):
    df_formato = df_formato.sort_values("gmv", ascending=False).reset_index(drop=True)
    df_formato["gmv_acumulado_pct"] = df_formato["gmv"].cumsum() / df_formato["gmv"].sum() * 100
    return df_formato

pareto_por_formato = {
    fmt: pareto(grp) for fmt, grp in gmv_por_categoria_formato.groupby("format")
}

for fmt, df_p in pareto_por_formato.items():
    print(f"\n--- {fmt} ---")
    display(df_p[df_p["gmv_acumulado_pct"] <= 80])


In [ ]:
# Comparación visual: HIPERMERCADO vs DESCUENTO
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fmt in zip(axes, ["HIPERMERCADO", "DESCUENTO"]):
    if fmt in pareto_por_formato:
        df_p = pareto_por_formato[fmt]
        ax.bar(df_p["category"], df_p["gmv"])
        ax.set_title(f"GMV por categoría — {fmt}")
        ax.tick_params(axis="x", rotation=60)
plt.tight_layout()
plt.savefig("bloque3_visualizaciones/p2_pareto_hipermercado_vs_descuento.png", dpi=150)
plt.show()


**Hallazgos a completar:**
- Categorías que concentran el 80% del GMV (chain-wide): `[completar]`
- ¿Coinciden entre HIPERMERCADO y DESCUENTO?: `[completar sí/no]`
- Interpretación de negocio: `[ej. DESCUENTO indexa más a Alimentos/Limpieza (compra de
  necesidad), HIPERMERCADO indexa más a Electrónica/Hogar (compra de conveniencia/impulso)]`


---
## Parte A · Pregunta 3 — Cohortes de lealtad

**Pregunta:** ¿Las cohortes más recientes retienen mejor o peor que las antiguas?
¿El ticket promedio de los clientes retenidos crece con el tiempo? ¿Mayor caída en qué mes?

> Reutiliza el resultado de la **Query 3 del Bloque 1** (tabla pivoteada de cohortes).
> Impórtalo aquí como CSV exportado de BigQuery, o recalcula en pandas con el bloque
> de abajo si prefieres mantener todo en un solo lugar.


In [ ]:
# Recalculo en pandas (equivalente a la Query 3 de BigQuery) para graficar directamente
loyal_tx = tx_completed[(tx_completed["loyalty_card"] == True) & (tx_completed["customer_id"].notna())].copy()

primera_compra = loyal_tx.groupby("customer_id")["transaction_date"].min().dt.to_period("M").rename("mes_cohorte")
loyal_tx = loyal_tx.merge(primera_compra, on="customer_id", how="left")
loyal_tx["mes_transaccion"] = loyal_tx["transaction_date"].dt.to_period("M")
loyal_tx["month_offset"] = (loyal_tx["mes_transaccion"] - loyal_tx["mes_cohorte"]).apply(lambda x: x.n)

tamano_cohorte = primera_compra.value_counts().rename("clientes_cohorte")

retencion = (
    loyal_tx[loyal_tx["month_offset"].isin([0, 1, 2, 3, 6])]
    .groupby(["mes_cohorte", "month_offset"])
    .agg(clientes_activos=("customer_id", "nunique"), ticket_promedio=("total_amount", "mean"))
    .reset_index()
)

pivot_retencion = retencion.pivot(index="mes_cohorte", columns="month_offset", values="clientes_activos")
pivot_retencion = pivot_retencion.div(tamano_cohorte, axis=0) * 100
pivot_retencion.columns = [f"retencion_mes_{c}_pct" for c in pivot_retencion.columns]
pivot_retencion


In [ ]:
# Heatmap de retención (visual estándar de cohortes)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot_retencion, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax)
ax.set_title("Retención (%) por cohorte y mes")
plt.tight_layout()
plt.savefig("bloque3_visualizaciones/p3_heatmap_retencion.png", dpi=150)
plt.show()


In [ ]:
# Ticket promedio de clientes retenidos a lo largo del tiempo
pivot_ticket = retencion.pivot(index="mes_cohorte", columns="month_offset", values="ticket_promedio")
pivot_ticket.columns = [f"ticket_mes_{c}" for c in pivot_ticket.columns]

fig, ax = plt.subplots()
pivot_ticket.mean(axis=0).plot(kind="line", marker="o", ax=ax)
ax.set_title("Ticket promedio según antigüedad del cliente (promedio de todas las cohortes)")
ax.set_xlabel("Mes desde primera compra")
ax.set_ylabel("Ticket promedio")
plt.tight_layout()
plt.savefig("bloque3_visualizaciones/p3_ticket_promedio_por_antiguedad.png", dpi=150)
plt.show()


**Hallazgos a completar:**
- Cohortes recientes vs antiguas: `[completar: retienen mejor/peor]`
- Tendencia del ticket promedio con el tiempo: `[crece/decrece] en [X]%`
- Mes con mayor caída de retención: `[completar]` — hipótesis: `[ej. fin de la novedad
  del programa de lealtad, o fin de una promoción de bienvenida]`


---
## Parte A · Pregunta 4 — Quiebres de stock y su impacto

**Pregunta:** ¿Hay categorías o proveedores donde los quiebres son sistemáticos?
¿Cuánto GMV se perdió? ¿Es un problema de demanda o de abastecimiento?

> Importa aquí el resultado de la **Query 5 del Bloque 1** (exportado de BigQuery).


In [ ]:
# TODO: reemplazar por el CSV real exportado de la Query 5
quiebres = pd.read_csv(f"{DATA_DIR}/query5_quiebres_stock.csv", parse_dates=["gap_inicio", "gap_fin"])

resumen_categoria = (
    quiebres.groupby("category", as_index=False)
    .agg(num_quiebres=("item_id", "count"), gmv_perdido_total=("gmv_estimado_perdido", "sum"),
         duracion_promedio=("duracion_dias", "mean"))
    .sort_values("gmv_perdido_total", ascending=False)
)
resumen_categoria


In [ ]:
fig, ax = plt.subplots()
ax.barh(resumen_categoria["category"], resumen_categoria["gmv_perdido_total"])
ax.set_title("GMV estimado perdido por quiebres, por categoría")
ax.set_xlabel("GMV perdido")
plt.tight_layout()
plt.savefig("bloque3_visualizaciones/p4_gmv_perdido_por_categoria.png", dpi=150)
plt.show()

print(f"GMV total estimado perdido por quiebres: {quiebres['gmv_estimado_perdido'].sum():,.2f}")


**Hallazgos a completar:**
- Categoría/proveedor con quiebres sistemáticos: `[completar]`
- GMV total perdido estimado: `$[completar]`
- Demanda vs. abastecimiento: `[si el quiebre ocurre justo después de picos de venta →
  problema de reabastecimiento/logística; si es aleatorio y disperso → posible problema
  de forecast de demanda]`


---
## Parte A · Pregunta 5 — Hallazgo libre

**Pregunta:** Identifica un hallazgo relevante no cubierto arriba, con evidencia
y explica el impacto de negocio.

**Sugerencias de ángulos a explorar (elige el que muestre el patrón más fuerte en tus datos):**
- Mix de `payment_method` (CASH/CARD/DIGITAL) por país o formato — ¿hay una tendencia
  hacia pagos digitales que sugiera invertir en esa infraestructura?
- Relación entre `is_shared_catalog` de vendors y velocidad de venta — ¿los productos
  de catálogo compartido rotan más rápido?
- Tasa de `RETURNED` por categoría o tienda — ¿hay una categoría con devoluciones
  anormalmente altas que sugiera un problema de calidad de producto?


In [ ]:
# Ejemplo: mix de payment_method por país (ajustar según el hallazgo elegido)
mix_pago = (
    tx_completed.merge(stores[["store_id", "country"]], on="store_id", how="left")
    .groupby(["country", "payment_method"]).size().reset_index(name="num_transacciones")
)
mix_pago_pct = mix_pago.pivot(index="country", columns="payment_method", values="num_transacciones")
mix_pago_pct = mix_pago_pct.div(mix_pago_pct.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots()
mix_pago_pct.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Mix de método de pago por país (%)")
ax.set_ylabel("% de transacciones")
plt.tight_layout()
plt.savefig("bloque3_visualizaciones/p5_hallazgo_libre.png", dpi=150)
plt.show()


**Hallazgo a completar:** `[describe el patrón encontrado, con el número que lo respalda]`
**Impacto de negocio:** `[qué debería hacer la empresa con este hallazgo]`


---
## Parte B · Interpretación de A/B Test

**Escenario:** Test de exhibición en punto de venta, 6 semanas (sept–oct 2024),
`CONTROL` vs `TREATMENT`, asignación aleatoria de tiendas.


In [ ]:
# TODO: ajustar rango de fechas según el test real
TEST_START = "2024-09-01"
TEST_END = "2024-10-31"
PRE_TEST_START = "2024-07-01"  # ventana pre-test para validar balance

ab_stores = store_promotions[store_promotions["promo_type"].notna()].copy()  # ajustar filtro por promo_name real
# Excluir tiendas contaminadas (asignadas a ambos variants) — detectadas en Bloque 0
contaminadas = (
    ab_stores.groupby("store_id")["variant"].nunique()
    .loc[lambda s: s > 1].index.tolist()
)
print(f"Tiendas excluidas por asignación duplicada: {len(contaminadas)}")

ab_stores_limpio = ab_stores[~ab_stores["store_id"].isin(contaminadas)][["store_id", "variant"]].drop_duplicates()
ab_stores_limpio = ab_stores_limpio.merge(stores, on="store_id", how="left")


### 1. Validación del experimento — ¿los grupos son comparables?

In [ ]:
pre_test_tx = tx_completed[
    (tx_completed["transaction_date"] >= PRE_TEST_START) &
    (tx_completed["transaction_date"] < TEST_START)
]
pre_test_gmv = pre_test_tx.groupby("store_id")["total_amount"].sum().rename("gmv_pre_test")

balance = ab_stores_limpio.merge(pre_test_gmv, on="store_id", how="left")

print("Balance de GMV pre-test por grupo:")
display(balance.groupby("variant")["gmv_pre_test"].agg(["count", "mean", "std"]))

print("\nBalance de formato por grupo (conteo):")
display(pd.crosstab(balance["variant"], balance["format"]))

print("\nBalance de tamaño (size_sqm) por grupo:")
display(balance.groupby("variant")["size_sqm"].agg(["mean", "std"]))

# Test formal de balance pre-test (importante: si esto da p < 0.05, el experimento
# tiene un problema de aleatorización que hay que reportar como limitación)
control_pre = balance[balance["variant"] == "CONTROL"]["gmv_pre_test"].dropna()
treat_pre = balance[balance["variant"] == "TREATMENT"]["gmv_pre_test"].dropna()
t_stat_pre, p_val_pre = stats.ttest_ind(control_pre, treat_pre, equal_var=False)
print(f"\nT-test de balance pre-test: t={t_stat_pre:.3f}, p-value={p_val_pre:.4f}")
print("Si p > 0.05: los grupos eran comparables ANTES del test (buena señal de aleatorización).")


### 2. Resultado en GMV — ¿TREATMENT genera más GMV semanal por tienda?

In [ ]:
test_tx = tx_completed[
    (tx_completed["transaction_date"] >= TEST_START) &
    (tx_completed["transaction_date"] <= TEST_END)
].copy()
test_tx["week"] = test_tx["transaction_date"].dt.to_period("W")

gmv_semanal_tienda = (
    test_tx.groupby(["store_id", "week"], as_index=False)["total_amount"].sum()
    .rename(columns={"total_amount": "gmv_semanal"})
    .merge(ab_stores_limpio[["store_id", "variant"]], on="store_id", how="inner")
)

control = gmv_semanal_tienda[gmv_semanal_tienda["variant"] == "CONTROL"]["gmv_semanal"]
treatment = gmv_semanal_tienda[gmv_semanal_tienda["variant"] == "TREATMENT"]["gmv_semanal"]

# Welch's t-test: no asumimos varianzas iguales (tiendas de tamaños distintos)
t_stat, p_value = stats.ttest_ind(treatment, control, equal_var=False)

diff_absoluta = treatment.mean() - control.mean()
lift_relativo = diff_absoluta / control.mean() * 100

# Intervalo de confianza 95% para la diferencia de medias
se_diff = np.sqrt(treatment.var(ddof=1)/len(treatment) + control.var(ddof=1)/len(control))
ci_95 = (diff_absoluta - 1.96*se_diff, diff_absoluta + 1.96*se_diff)

# Effect size (Cohen's d, con desviación estándar combinada)
pooled_std = np.sqrt(((len(treatment)-1)*treatment.var(ddof=1) + (len(control)-1)*control.var(ddof=1)) /
                      (len(treatment) + len(control) - 2))
cohens_d = diff_absoluta / pooled_std

print(f"GMV semanal promedio — CONTROL:   {control.mean():,.2f}")
print(f"GMV semanal promedio — TREATMENT: {treatment.mean():,.2f}")
print(f"Diferencia absoluta: {diff_absoluta:,.2f}")
print(f"Lift relativo: {lift_relativo:.2f}%")
print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.4f}")
print(f"IC 95% de la diferencia: [{ci_95[0]:,.2f}, {ci_95[1]:,.2f}]")
print(f"Cohen's d (tamaño del efecto): {cohens_d:.3f}")


In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=gmv_semanal_tienda, x="variant", y="gmv_semanal", ax=ax)
ax.set_title("Distribución de GMV semanal por tienda — CONTROL vs TREATMENT")
plt.tight_layout()
plt.savefig("bloque3_visualizaciones/ab_test_gmv_boxplot.png", dpi=150)
plt.show()


### 3. Resultado en ticket y frecuencia — ¿de dónde viene el efecto?

In [ ]:
test_items = test_tx.merge(ab_stores_limpio[["store_id", "variant"]], on="store_id", how="inner")

metricas_por_variant = test_items.groupby("variant").agg(
    ticket_promedio=("total_amount", "mean"),
    num_transacciones=("transaction_id", "nunique"),
    num_tiendas=("store_id", "nunique"),
).assign(transacciones_por_tienda=lambda d: d["num_transacciones"] / d["num_tiendas"])

metricas_por_variant


In [ ]:
# T-test separado para ticket promedio (a nivel transacción)
ticket_control = test_tx.merge(ab_stores_limpio[["store_id","variant"]], on="store_id")\
    .query("variant == 'CONTROL'")["total_amount"]
ticket_treatment = test_tx.merge(ab_stores_limpio[["store_id","variant"]], on="store_id")\
    .query("variant == 'TREATMENT'")["total_amount"]

t_ticket, p_ticket = stats.ttest_ind(ticket_treatment, ticket_control, equal_var=False)
print(f"Ticket promedio — diferencia: {ticket_treatment.mean() - ticket_control.mean():,.2f}, p-value: {p_ticket:.4f}")

# Interpretación: si el ticket NO cambia significativamente pero las transacciones/tienda sí,
# el efecto viene del TRÁFICO (más compras), no del monto por compra — y viceversa.


**Hallazgos a completar:**
- El efecto en GMV viene principalmente de: `[completar: ticket más alto / más
  transacciones / ambos]`, según cuál de los dos t-tests (GMV total vs. ticket) dio
  significativo.


### 4. Decisión de negocio

**Framework de decisión (completar con los valores reales de arriba):**

| Criterio | Valor | Interpretación |
|---|---|---|
| P-value | `[completar]` | `[< 0.05 → significativo / entre 0.05-0.10 → marginal / > 0.10 → no significativo]` |
| Lift relativo | `[completar]%` | `[tamaño del efecto en términos de negocio]` |
| Cohen's d | `[completar]` | `[< 0.2 pequeño / 0.2-0.5 mediano / > 0.5 grande]` |
| Costo de implementación | `[completar]` | Costo de replicar la exhibición en todas las tiendas |

**Recomendación:** `[completar según los datos: implementar / no implementar / extender el test]`

**Nota especial — si el p-value fuera 0.08 (caso del enunciado):**
Un p-value de 0.08 está por encima del umbral convencional de 0.05, pero no significa
"no hay efecto" — significa que la evidencia no es concluyente con el tamaño de
muestra actual (número de tiendas en el test, no de transacciones). La decisión correcta
no es un sí/no automático:
1. Revisar el **poder estadístico** del test — con pocas tiendas, incluso un efecto real
   puede no alcanzar significancia al 95%.
2. Comparar el **lift observado** contra el costo de implementación — si el lift es
   grande y el costo de montar la exhibición es bajo, el riesgo de un falso positivo
   se puede asumir (no todas las decisiones de negocio requieren 95% de confianza).
3. Considerar **extender el test 2-4 semanas más** en vez de decidir con datos
   insuficientes, especialmente si el costo de esperar es bajo comparado con el de
   un rollout equivocado a toda la cadena.
